In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Replication: Iterative Inference in a Chess-Playing Neural Network

## Overview

This notebook replicates key experiments from the "Iterative Inference in a Chess-Playing Neural Network" paper, which investigates how neural networks progressively build understanding across layers using the logit lens technique applied to Leela Chess Zero.

## Key Hypotheses to Test
1. Neural networks perform iterative inference with capability progression occurring in distinct computational phases
2. The inference process shows a three-phase pattern: early rapid gains, middle plateau, late sharp strengthening

## Experiments to Replicate
1. **Demo with Logit Lens** - Visualize layer-wise policy evolution on sample positions
2. **Policy Distribution Metrics** - JS divergence, entropy, Kendall's tau correlation
3. **Probability Evolution** - Track how move probabilities change across layers

In [2]:
# Set up the working directory and paths
import sys
import os

# Set working directory to the repository
REPO_ROOT = "/net/scratch2/smallyan/leela_eval"
os.chdir(REPO_ROOT)

# Add src to path for imports
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

print(f"Working directory: {os.getcwd()}")
print(f"Repository root: {REPO_ROOT}")

Working directory: /net/scratch2/smallyan/leela_eval
Repository root: /net/scratch2/smallyan/leela_eval


In [3]:
# Check CUDA availability and set device
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("CUDA not available, using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

Using CUDA device: NVIDIA H200 NVL
PyTorch version: 2.7.1+cu118
Device: cuda


In [4]:
# Set random seed for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
print(f"Random seed set to {SEED}")

Random seed set to 42


## Part 1: Load Model and Initialize Logit Lens

We load the Leela Chess Zero model (lc0-original.onnx) and initialize the logit lens for layer-wise analysis.

In [5]:
# Import the main components from leela_logit_lens package
from leela_interp import Lc0sight, LeelaBoard
from leela_logit_lens import LeelaLogitLens

# Model path - using the original (non-finetuned) model
model_path = os.path.join(REPO_ROOT, "lc0-original.onnx")
print(f"Loading model from: {model_path}")

# Initialize the model with GPU
model = Lc0sight(path=model_path, device=device)
model.eval()
print(f"Model loaded successfully on device: {device}")

# Initialize the logit lens
lens = LeelaLogitLens(model)
print(f"LeelaLogitLens initialized")
print(f"Number of layers: {lens.num_layers}")
print(f"Hidden dimension: {lens.hidden_dim}")
print(f"Number of tokens (squares): {lens.num_tokens}")

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading model from: /net/scratch2/smallyan/leela_eval/lc0-original.onnx
Using device: cuda


AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): Onnx

In [6]:
# Reload the modules after reinstallation
import importlib
import sys

# Remove cached modules to force reload
modules_to_remove = [key for key in sys.modules.keys() if 'leela' in key]
for mod in modules_to_remove:
    del sys.modules[mod]

# Re-import
from leela_interp import Lc0sight, LeelaBoard
from leela_logit_lens import LeelaLogitLens

# Model path - using the original (non-finetuned) model
model_path = os.path.join(REPO_ROOT, "lc0-original.onnx")
print(f"Loading model from: {model_path}")

# Initialize the model with GPU
model = Lc0sight(path=model_path, device=device)
model.eval()
print(f"Model loaded successfully")

# Initialize the logit lens
lens = LeelaLogitLens(model)
print(f"LeelaLogitLens initialized")
print(f"Number of layers: {lens.num_layers}")
print(f"Hidden dimension: {lens.hidden_dim}")

Loading model from: /net/scratch2/smallyan/leela_eval/lc0-original.onnx
Using device: cuda


AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): Onnx

In [7]:
# Restart kernel-style cleanup and reload
import importlib
import sys

# Clear all leela-related modules
modules_to_remove = [key for key in list(sys.modules.keys()) if 'leela' in key.lower() or 'nnsight' in key.lower()]
for mod in modules_to_remove:
    try:
        del sys.modules[mod]
    except:
        pass

print("Modules cleaned, reloading...")

Modules cleaned, reloading...


In [8]:
# Re-import after cleaning modules
from leela_interp import Lc0sight, LeelaBoard
from leela_logit_lens import LeelaLogitLens

# Model path - using the original (non-finetuned) model
REPO_ROOT = "/net/scratch2/smallyan/leela_eval"
model_path = os.path.join(REPO_ROOT, "lc0-original.onnx")
print(f"Loading model from: {model_path}")

# Initialize the model with GPU
model = Lc0sight(path=model_path, device="cuda")
print(f"Model loaded successfully")

# Initialize the logit lens
lens = LeelaLogitLens(model)
print(f"LeelaLogitLens initialized")
print(f"Number of layers: {lens.num_layers}")
print(f"Hidden dimension: {lens.hidden_dim}")

Loading model from: /net/scratch2/smallyan/leela_eval/lc0-original.onnx
Using device: cuda


AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): Onnx